In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/raw/'
PROCESSED_PATH = '../data/processed/'

N_WEEKS = 8  # Number of weeks to use as sequence length

print('Config set.')
print(f'Using first {N_WEEKS} weeks of each student.')

Config set.
Using first 8 weeks of each student.


## Step 1 - Load Data

In [2]:
student_info = pd.read_csv(DATA_PATH + 'studentInfo.csv')
student_ass  = pd.read_csv(DATA_PATH + 'studentAssessment.csv')
assessments  = pd.read_csv(DATA_PATH + 'assessments.csv')
registration = pd.read_csv(DATA_PATH + 'studentRegistration.csv')
student_vle  = pd.read_csv(DATA_PATH + 'studentVle.csv') 

print(f'Students loaded: {len(student_info)}')

Students loaded: 32593


## Step 2 - Create Labels

In [3]:
# Binary label: 0 = Pass/Distinction, 1 = Fail/Withdrawn
student_info['label'] = student_info['final_result'].apply(
    lambda x: 0 if x in ['Pass', 'Distinction'] else 1)

print('Label created:')
print(student_info['label'].value_counts())
print(f'At-risk rate: {student_info["label"].mean()*100:.1f}%')

Label created:
label
1    17208
0    15385
Name: count, dtype: int64
At-risk rate: 52.8%


## Step 3 - Prepare Assessment Data

In [4]:
# Merge assessment scores with assessment metadata
ass_full = student_ass.merge(
    assessments[['id_assessment', 'date', 'assessment_type', 'weight']],
    on='id_assessment', how='left')

# Remove rows where assessment date is missing (exam entries)
ass_full = ass_full.dropna(subset=['date'])

# Convert day number to week number
ass_full['week'] = (ass_full['date'] / 7).astype(int) + 1

# Fill missing scores with 0 (student did not submit = 0)
ass_full['score'] = ass_full['score'].fillna(0)

# Calculate if submitted on time
# date = due date, date_submitted = when they submitted
ass_full['on_time'] = (ass_full['date_submitted'] <= ass_full['date']).astype(int)

print('Assessment data prepared:')
print(f'Total records: {len(ass_full)}')
print(f'Week range: {ass_full["week"].min()} to {ass_full["week"].max()}')

Assessment data prepared:
Total records: 171047
Week range: 2 to 38


## Step 4 - Extract Weekly Features Per Student

In [7]:
def extract_weekly_features(student_id, module,
                              presentation, ass_data,
                              vle_data, n_weeks=8):
    s_ass = ass_data[
        (ass_data['id_student'] == student_id) &
        (ass_data['week'] <= n_weeks)
    ].copy()

    if s_ass['week'].nunique() < 2:
        return None

    # VLE data — always available now
    s_vle = vle_data[
        (vle_data['id_student'] == student_id) &
        (vle_data['code_module'] == module) &
        (vle_data['code_presentation'] == presentation)
    ].copy()
    s_vle['week'] = (s_vle['date'] / 7).astype(int) + 1
    s_vle = s_vle[s_vle['week'] <= n_weeks]

    running_scores = []
    prev_week_score = 50.0
    weekly_features = []

    for week in range(1, n_weeks + 1):
        week_ass = s_ass[s_ass['week'] == week]
        week_vle = s_vle[s_vle['week'] == week]

        f1_login  = len(week_vle)
        f2_clicks = week_vle['sum_click'].sum() \
                    if len(week_vle) > 0 else 0

        f3_score  = week_ass['score'].mean() \
                    if len(week_ass) > 0 else prev_week_score

        f4_num_ass = len(week_ass)

        f5_ontime  = week_ass['on_time'].mean() \
                     if len(week_ass) > 0 else 1.0

        f6_inactive = max(0, 7 - week_ass[
            'date_submitted'].nunique()) \
            if len(week_ass) > 0 else 7.0

        f7_trend = f3_score - prev_week_score
        prev_week_score = f3_score

        running_scores.append(f3_score)
        f8_cumulative = np.mean(running_scores)

        weekly_features.append([
            f1_login, f2_clicks, f3_score,
            f4_num_ass, f5_ontime, f6_inactive,
            f7_trend, f8_cumulative
        ])

    return np.array(weekly_features, dtype=np.float32)

## Step 5 - Run Feature Extraction on All Students

In [10]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

print("Fast vectorised feature extraction...")

# ── Step 1: Prepare VLE grouped by student+week ──────────
print("Step 1/5: Grouping VLE data...")
vle = student_vle.copy()
vle['week'] = (vle['date'] / 7).astype(int) + 1
vle = vle[vle['week'] <= N_WEEKS]

vle_grp = vle.groupby(
    ['id_student','code_module',
     'code_presentation','week'],
    as_index=False).agg(
    f1_login  =('sum_click','count'),
    f2_clicks =('sum_click','sum'))

print(f"  VLE grouped: {len(vle_grp)} rows")

# ── Step 2: Prepare assessment grouped by student+week ───
print("Step 2/5: Grouping assessment data...")
ass = ass_full.copy()
ass = ass[ass['week'] <= N_WEEKS]

ass_grp = ass.groupby(
    ['id_student','week'],
    as_index=False).agg(
    f3_score    =('score','mean'),
    f4_num_ass  =('id_assessment','count'),
    f5_ontime   =('on_time','mean'),
    active_days =('date_submitted','nunique'))

ass_grp['f6_inactive'] = (
    7 - ass_grp['active_days']).clip(lower=0)

print(f"  Assessment grouped: {len(ass_grp)} rows")

# ── Step 3: Build full grid student x week ───────────────
print("Step 3/5: Building student-week grid...")
students = student_info[
    ['id_student','code_module',
     'code_presentation','label']
].drop_duplicates().reset_index(drop=True)

# Find students with at least 2 weeks of assessment data
valid_ids = ass_grp[ass_grp['f4_num_ass']>0]\
    .groupby('id_student')['week']\
    .nunique()
valid_ids = valid_ids[valid_ids >= 2].index
students  = students[
    students['id_student'].isin(valid_ids)]

print(f"  Valid students: {len(students)}")

# Create week numbers
all_weeks = pd.DataFrame({'week': range(1, N_WEEKS+1)})

# Cross join — every student x every week
grid = students.merge(all_weeks, how='cross')
print(f"  Grid size: {len(grid)} rows")

# ── Step 4: Merge all features ───────────────────────────
print("Step 4/5: Merging features...")

# Merge assessment features
grid = grid.merge(
    ass_grp[['id_student','week','f3_score',
              'f4_num_ass','f5_ontime','f6_inactive']],
    on=['id_student','week'], how='left')

# Merge VLE features
grid = grid.merge(
    vle_grp[['id_student','code_module',
              'code_presentation','week',
              'f1_login','f2_clicks']],
    on=['id_student','code_module',
        'code_presentation','week'],
    how='left')

# Sort
grid = grid.sort_values(
    ['id_student','week']).reset_index(drop=True)

# Fill missing values
grid['f1_login']    = grid['f1_login'].fillna(0)
grid['f2_clicks']   = grid['f2_clicks'].fillna(0)
grid['f4_num_ass']  = grid['f4_num_ass'].fillna(0)
grid['f5_ontime']   = grid['f5_ontime'].fillna(1.0)
grid['f6_inactive'] = grid['f6_inactive'].fillna(7.0)

# Forward fill score (carry last known score forward)
grid['f3_score'] = grid.groupby('id_student')[
    'f3_score'].transform(
    lambda x: x.fillna(method='ffill').fillna(50.0))

# Calculate F7 trend and F8 cumulative
grid['f7_trend'] = grid.groupby('id_student')[
    'f3_score'].diff().fillna(0)

grid['f8_cumulative'] = grid.groupby('id_student')[
    'f3_score'].transform(
    lambda x: x.expanding().mean())

print(f"  Features merged. Grid: {len(grid)} rows")

# ── Step 5: Build numpy arrays ───────────────────────────
print("Step 5/5: Building X and y arrays...")

feature_cols = [
    'f1_login','f2_clicks','f3_score',
    'f4_num_ass','f5_ontime','f6_inactive',
    'f7_trend','f8_cumulative']

# Only keep students with exactly N_WEEKS rows
counts = grid.groupby('id_student')['week'].count()
full_students = counts[counts == N_WEEKS].index
grid = grid[grid['id_student'].isin(full_students)]

# Reshape to 3D array
n_students = len(full_students)
X = grid[feature_cols].values.reshape(
    n_students, N_WEEKS, len(feature_cols))
X = X.astype(np.float32)

# Get labels in same order
labels = grid.groupby('id_student')[
    'label'].first()
labels = labels.loc[full_students]
y = labels.values.astype(np.float32)

# Clean NaN and Inf
X = np.nan_to_num(X, nan=0.0,
                   posinf=100.0, neginf=0.0)

print(f'\n=== FEATURE EXTRACTION COMPLETE ===')
print(f'Students with features: {len(X):,}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'At-risk: {y.mean()*100:.1f}%')


Fast vectorised feature extraction...
Step 1/5: Grouping VLE data...
  VLE grouped: 205770 rows
Step 2/5: Grouping assessment data...
  Assessment grouped: 43998 rows
Step 3/5: Building student-week grid...
  Valid students: 20163
  Grid size: 161304 rows
Step 4/5: Merging features...
  Features merged. Grid: 161304 rows
Step 5/5: Building X and y arrays...

=== FEATURE EXTRACTION COMPLETE ===
Students with features: 14,631
X shape: (14631, 8, 8)
y shape: (14631,)
At-risk: 32.3%


## Step 6 - Quality Check

In [11]:
print('=== DATA QUALITY CHECK ===')
print(f'NaN values in X: {np.isnan(X).sum()}')
print(f'Inf values in X: {np.isinf(X).sum()}')

X = np.nan_to_num(X, nan=0.0,
                   posinf=100.0, neginf=0.0)

print(f'After cleaning:')
print(f'NaN values: {np.isnan(X).sum()}')
print(f'Inf values: {np.isinf(X).sum()}')
print()
print('=== FEATURE STATISTICS ===')
feat_names = ['F1_logins','F2_clicks','F3_score',
              'F4_num_ass','F5_ontime','F6_inactive',
              'F7_trend','F8_cumulative']
for i, fname in enumerate(feat_names):
    vals = X[:, :, i].flatten()
    print(f'  {fname:<16}: mean={vals.mean():.2f} '
          f'std={vals.std():.2f} '
          f'min={vals.min():.2f} '
          f'max={vals.max():.2f}')

=== DATA QUALITY CHECK ===
NaN values in X: 0
Inf values in X: 0
After cleaning:
NaN values: 0
Inf values: 0

=== FEATURE STATISTICS ===
  F1_logins       : mean=21.77 std=22.31 min=0.00 max=365.00
  F2_clicks       : mean=78.61 std=115.06 min=0.00 max=6999.00
  F3_score        : mean=67.90 std=21.09 min=0.00 max=100.00
  F4_num_ass      : mean=0.29 std=0.48 min=0.00 max=2.00
  F5_ontime       : mean=0.93 std=0.25 min=0.00 max=1.00
  F6_inactive     : mean=6.71 std=0.47 min=5.00 max=7.00
  F7_trend        : mean=3.64 std=13.89 min=-90.00 max=100.00
  F8_cumulative   : mean=59.87 std=12.52 min=12.50 max=92.50


## Step 7 - Save Files 

In [12]:
import os
os.makedirs(PROCESSED_PATH, exist_ok=True)

np.save(PROCESSED_PATH + 'X_sequences.npy', X)
np.save(PROCESSED_PATH + 'y_labels.npy', y)

print('=== SAVED ===')
print(f'X_sequences.npy → {X.shape}')
print(f'y_labels.npy    → {y.shape}')
print()
print('Notebook 2 complete.')
print('Ready to run model files.')

=== SAVED ===
X_sequences.npy → (14631, 8, 8)
y_labels.npy    → (14631,)

Notebook 2 complete.
Ready to run model files.
